# MUSE Visualization Notebook
All visualization tools in one place. Edit the **Input paths** cell and run top-to-bottom.

In [ ]:
# ── Input paths ─────────────────────────────────────────────────────────────
MODEL     = "/path/to/model.pdb"          # PDB or mmCIF
MAP_ED    = "/path/to/2fofc.ccp4"         # electron density (2Fo-Fc)
MAP_SNR   = "/path/to/snr.ccp4"           # SNR map
MAP_PVAL  = "/path/to/pvalue.ccp4"        # p-value map  [0, 1]
RESOLUTION = 2.0                          # Å

# ── Per-analysis settings ────────────────────────────────────────────────────
CHAIN      = "A"                          # chain for profile / water plots
LIG_SEQ_ID = 401                          # residue seq-id of the ligand
LIG_CHAIN  = "A"                          # chain of the ligand

WATER_THRESHOLD = 0.4                     # remove waters with p-value MUSEm < this
OUT_DIR    = "/path/to/output"            # folder for all saved figures / PDBs

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('')))
os.makedirs(OUT_DIR, exist_ok=True)

import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from muse import (
    run_muse,
    export_atom_csv, export_residue_csv,
    electron_density_config, snr_map_config, pvalue_map_config,
    extract_residue_scores, extract_bfactors,
    write_scored_pdb,
    plot_residue_profile,
    plot_ligand_density_support,
    plot_water_support,
    filter_waters_by_score,
)

print("Imports OK")

## 1 — Run MUSE for all three map types

In [ ]:
# Electron-density map  (normalize=True, zeta=1.2 — original EDIA parameters)
result_ed = run_muse(
    map_path       = MAP_ED,
    structure_path = MODEL,
    resolution     = RESOLUTION,
    config         = electron_density_config(),
)
print(f"ED   → {len(result_ed.atom_scores)} atoms, "
      f"{len(result_ed.residue_scores)} residues, OPIA={result_ed.opia:.3f}")

In [ ]:
# SNR map  (normalize=False, zeta=5.0 — raw values, higher truncation ceiling)
result_snr = run_muse(
    map_path       = MAP_SNR,
    structure_path = MODEL,
    resolution     = RESOLUTION,
    config         = snr_map_config(zeta=5.0),
)
print(f"SNR  → {len(result_snr.atom_scores)} atoms, "
      f"{len(result_snr.residue_scores)} residues")

In [ ]:
# P-value map  (normalize=False, zeta=1.0 — natural [0,1] probability range)
result_pv = run_muse(
    map_path       = MAP_PVAL,
    structure_path = MODEL,
    resolution     = RESOLUTION,
    config         = pvalue_map_config(),
)
print(f"PVal → {len(result_pv.atom_scores)} atoms, "
      f"{len(result_pv.residue_scores)} residues")

## 2 — Export CSV scores

In [ ]:
for tag, result in [("ed", result_ed), ("snr", result_snr), ("pval", result_pv)]:
    export_atom_csv    (result, f"{OUT_DIR}/atoms_{tag}.csv")
    export_residue_csv (result, f"{OUT_DIR}/residues_{tag}.csv")
    print(f"Saved {tag} CSVs")

## 3 — Scored PDB for PyMOL / ChimeraX colouring

```
PyMOL:    spectrum b, blue_white_red, model_ed_residue,  minimum=0, maximum=120
ChimeraX: color bfactor #1  palette blue:white:red  range 0,120
```

In [ ]:
# Per-residue MUSEm — electron density (scale ×100 → [0, 120] range in B-column)
write_scored_pdb(
    result         = result_ed,
    structure_path = MODEL,
    output_path    = f"{OUT_DIR}/model_ed_residue.pdb",
    score_level    = "residue",   # all atoms of a residue share its MUSEm
    score_field    = "musem",
    score_scale    = 100.0,
)

# Per-atom score — electron density  (finer granularity)
write_scored_pdb(
    result         = result_ed,
    structure_path = MODEL,
    output_path    = f"{OUT_DIR}/model_ed_atom.pdb",
    score_level    = "atom",
    score_field    = "score_positive",   # MUSE+
    score_scale    = 100.0,
)

# Per-residue MUSEm — p-value map  (scale ×100 → [0, 100])
write_scored_pdb(
    result         = result_pv,
    structure_path = MODEL,
    output_path    = f"{OUT_DIR}/model_pval_residue.pdb",
    score_level    = "residue",
    score_field    = "musem",
    score_scale    = 100.0,
)

print("Scored PDBs written.")

## 4 — Residue profile strip chart (dual Y-axis)

In [ ]:
# ── Build per-residue data series ────────────────────────────────────────────
ds_edia   = extract_residue_scores(result_ed,  score_field="musem", chain_id=CHAIN)
ds_snr    = extract_residue_scores(result_snr, score_field="musem", chain_id=CHAIN)
ds_pval   = extract_residue_scores(result_pv,  score_field="musem", chain_id=CHAIN)
ds_bfac   = extract_bfactors(MODEL, chain_id=CHAIN, atom_name="CA")

fig = plot_residue_profile(
    datasets = {
        "EDIA"    : ds_edia,
        "B-factor": ds_bfac,
        "SNR"     : ds_snr,
        "p-value" : ds_pval,
    },

    # ── Axis assignment ──────────────────────────────────────────────────────
    left_series  = ["EDIA", "B-factor"],  # filled area + line on left axis
    right_series = ["SNR"],               # dashed line on right axis

    # ── Background bin colouring (p-value MUSEm) ─────────────────────────────
    bin_color_series = "p-value",
    bin_colormap     = "RdYlGn",  # red=low confidence, green=high
    bin_vmin         = 0.0,
    bin_vmax         = 1.0,
    colorbar_label   = "p-value MUSEm",

    # ── Per-series display transforms ────────────────────────────────────────
    # SNR values can be large — compress with log(1+x)
    transforms = {"SNR": ("log1p", 1.0)},

    # ── Labels ───────────────────────────────────────────────────────────────
    left_ylabel  = "MUSEm (EDIA)  /  B-factor (Å²)",
    right_ylabel = "SNR  log(1 + MUSEm)",
    title        = f"Residue quality profile — chain {CHAIN}",
    figsize      = (16, 5),

    # Draw horizontal quality-threshold lines at 0.4 and 0.8 on the left axis
    edia_threshold_lines = True,
)

fig.savefig(f"{OUT_DIR}/residue_profile.png", dpi=150, bbox_inches="tight")
fig

In [ ]:
# ── Variant: EDIA vs SNR on left, p-value on right ──────────────────────────
fig2 = plot_residue_profile(
    datasets = {"EDIA": ds_edia, "SNR": ds_snr, "p-value": ds_pval},
    left_series      = ["EDIA", "SNR"],
    right_series     = ["p-value"],
    bin_color_series = "p-value",
    transforms       = {"SNR": ("log1p", 1.0)},
    left_ylabel      = "MUSEm",
    right_ylabel     = "p-value MUSEm",
    title            = f"EDIA + SNR vs p-value — chain {CHAIN}",
    edia_threshold_lines = True,
)
fig2.savefig(f"{OUT_DIR}/residue_profile_variant.png", dpi=150, bbox_inches="tight")
fig2

## 5 — Small-molecule / ligand density contour plot

In [ ]:
# Extract atom scores for the target ligand residue from the ED result
lig_scores_ed = [
    a for a in result_ed.atom_scores
    if a.chain_id == LIG_CHAIN and a.residue_seq_id == LIG_SEQ_ID
]
print(f"Found {len(lig_scores_ed)} atoms for residue {LIG_CHAIN}/{LIG_SEQ_ID}")

# 2D layout is computed automatically from the PDB topology using RDKit
# (AllChem.Compute2DCoords).  No SMILES string required.
# 'projection' is only used as fallback when RDKit is not installed.
fig_lig = plot_ligand_density_support(
    atom_scores    = lig_scores_ed,
    structure_path = MODEL,
    chain_id       = LIG_CHAIN,
    residue_seq_id = LIG_SEQ_ID,

    score_key      = "score_positive",  # MUSE+: supports positive density

    # Contour / circle thresholds (EDIA scale)
    good_threshold = 0.8,   # green blobs for well-supported atoms
    poor_threshold = 0.4,   # pink dashed circles for missing density

    # Visual parameters
    atom_sigma       = 0.7,   # Gaussian kernel width in Å
    n_contour_levels = 7,
    padding          = 2.0,
    figsize          = (7, 7),
    show_atom_labels = True,
    label_carbon     = False,
)

fig_lig.savefig(f"{OUT_DIR}/ligand_ed_{LIG_SEQ_ID}.png", dpi=150, bbox_inches="tight")
fig_lig

In [ ]:
# ── Same ligand — p-value map scores (uncertainty view) ─────────────────────
lig_scores_pv = [
    a for a in result_pv.atom_scores
    if a.chain_id == LIG_CHAIN and a.residue_seq_id == LIG_SEQ_ID
]

fig_lig_pv = plot_ligand_density_support(
    atom_scores    = lig_scores_pv,
    structure_path = MODEL,
    chain_id       = LIG_CHAIN,
    residue_seq_id = LIG_SEQ_ID,
    score_key      = "score_positive",
    title          = f"Res {LIG_SEQ_ID} — p-value map support",
)
fig_lig_pv.savefig(f"{OUT_DIR}/ligand_pval_{LIG_SEQ_ID}.png", dpi=150, bbox_inches="tight")
fig_lig_pv

In [ ]:
# ── Custom dict: pass any per-atom value (e.g. from an external tool) ───────
custom_scores = {
    # atom_name: value  — replace with your own data
    "C1" : 0.95,
    "N2" : 0.32,   # will get a dashed circle
    "O3" : 0.78,
    "C4" : 0.88,
    "S5" : 0.15,   # will get a dashed circle
}

fig_custom = plot_ligand_density_support(
    atom_scores    = custom_scores,   # dict, not a result object
    structure_path = MODEL,
    chain_id       = LIG_CHAIN,
    residue_seq_id = LIG_SEQ_ID,
    title          = f"Res {LIG_SEQ_ID} — custom scores",
)
fig_custom

## 6 — Water molecule analysis

In [ ]:
# Rank plot + histogram of all water p-value scores
fig_wat = plot_water_support(
    result      = result_pv,
    threshold   = WATER_THRESHOLD,
    chain_id    = None,          # None = all chains
    score_field = "musem",
    colormap    = "RdYlGn",
    title       = "Water p-value MUSEm support",

    # Label waters within ±0.1 of the threshold so you can identify boundary cases
    label_near_threshold = 0.08,
    max_label            = 40,
)

fig_wat.savefig(f"{OUT_DIR}/water_support.png", dpi=150, bbox_inches="tight")
fig_wat

In [ ]:
# Experiment with a stricter threshold before committing to filtering
for thresh in [0.3, 0.4, 0.5]:
    n_keep = sum(
        1 for r in result_pv.residue_scores
        if r.residue_name.upper() in {"HOH","WAT","DOD","H2O","SOL"}
        and r.musem_score >= thresh
    )
    n_all = sum(
        1 for r in result_pv.residue_scores
        if r.residue_name.upper() in {"HOH","WAT","DOD","H2O","SOL"}
    )
    print(f"threshold={thresh:.1f}  keep={n_keep}/{n_all}  remove={n_all-n_keep}")

## 7 — Write filtered model (waters removed)

In [ ]:
n_kept, n_removed = filter_waters_by_score(
    result         = result_pv,
    structure_path = MODEL,
    output_path    = f"{OUT_DIR}/model_waters_filtered.pdb",
    threshold      = WATER_THRESHOLD,
    score_field    = "musem",
    chain_id       = None,   # filter all chains; set e.g. "A" for one chain only
)

print(f"Filtered model written.")
print(f"  Waters kept   : {n_kept}")
print(f"  Waters removed: {n_removed}")
print(f"  Threshold     : {WATER_THRESHOLD}  (p-value MUSEm)")